# 第2回 発展課題：用語集の Q&A を LoRA で学習する

提供ノートブックとは独立した実験です。用語集の 58 項目から生成した Q&A を学習し、学習に使わない質問で前後を比較します。**実行結果は含まれていません**。

1. ローカルのリポジトリで `uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl` を実行します。
2. このノートブックを Colab にアップロードして開き、ランタイムを **T4 GPU** に変更します。
3. Colab のファイル欄で生成した `glossary_sft.jsonl` と `data/glossary.json` を `/content` にアップロードしてから、以下のセルを上から実行します。出力をダウンロードして `lectures/02/` に保存します。
4. 学習前・後の出力を同じ基準で手動採点し、実行結果と考察を `lectures/02/answer.md` に書きます。終わったらランタイムを削除します。

元データは assistant が定義文を返す形式で、FAQ 演習の JSON・`needs_review` 形式ではありません。本実験は元の Q&A 形式を使います。未収録語の確認要求は教師データに含まれないため、学習すればそう振る舞うと期待できません。20 epoch は演習と同じですが、データが 14 件から 116 件に増えるため学習時間も長くなります。GPU が使用できなければ、結果を作ったことにしないでください。

## 0. GPU とライブラリ

講義ノートブックの動作確認済みバージョンと同じ設定です。T4 以外や CPU ではこの実験を開始しません。

In [ ]:
import subprocess
info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(info.stdout or info.stderr)
if info.returncode != 0 or "T4" not in info.stdout:
    raise RuntimeError("T4 GPU を選択して、ランタイムを再起動してください。")

In [ ]:
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定とデータ

`glossary.json` は学習時にモデルへ渡さず、評価の正解としてのみ使います。train と評価質問の文字列が重複しないことを確認します。同じ用語の定義を学習しているので、評価は未知用語への汎化ではなく、**既知の用語に関する言い換え**の評価です。

In [ ]:
from pathlib import Path
import json, time, hashlib
from google.colab import files

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42, "num_train_epochs": 20, "warmup_steps": 6, "lr": 2e-4,
    "lora_r": 16, "lora_alpha": 32, "max_seq_len": 512,
    "max_new_tokens": 192, "gen": {"do_sample": False},
}
WORK = Path("/content/glossary_lora_results")
WORK.mkdir(parents=True, exist_ok=True)
print("Colab のファイル欄から glossary_sft.jsonl と glossary.json を /content にアップロードしてください。")
uploaded = {name: (Path("/content") / name).read_bytes() for name in ("glossary_sft.jsonl", "glossary.json")}
for name in ("glossary_sft.jsonl", "glossary.json"):
    if name not in uploaded:
        raise ValueError(f"{name} が必要です。このセルを再実行して両方選択してください。")
train = [json.loads(line) for line in uploaded["glossary_sft.jsonl"].decode("utf-8").splitlines() if line.strip()]
glossary = json.loads(uploaded["glossary.json"].decode("utf-8"))["terms"]
by_id = {entry["id"]: entry for entry in glossary}
assert train and all([m["role"] for m in x["messages"]] == ["user", "assistant"] for x in train)
print("学習データ:", len(train), "件 / 用語:", len(glossary), "件")
print("SHA256:", {name: hashlib.sha256(uploaded[name]).hexdigest() for name in ("glossary_sft.jsonl", "glossary.json")})

In [ ]:
# 固有名を含む質問、別名・言い換え質問、用語集にない架空語を用意する。
# None は用語集に該当項目がないことを示す（一般世界にも存在しないことの証明ではない）。
specs = [
    ("known", "lora", "LoRAの仕組みを説明してください。"),
    ("known", "token", "トークンとはどのようなものですか？"),
    ("known", "overfitting", "過学習はどんな状態ですか？"),
    ("known", "mcp", "MCPについて教えてください。"),
    ("known", "api-key", "APIキーの意味を教えてください。"),
    ("paraphrase", "lora", "Low-Rank Adaptationはどんな方法ですか？"),
    ("paraphrase", "context-window", "コンテキスト長が表す上限を教えて。"),
    ("paraphrase", "sft", "SFTはどのようなデータを使いますか？"),
    ("paraphrase", "hallucination", "AIの幻覚とは何を指しますか？"),
    ("paraphrase", "epoch", "学習データを1周する単位を説明してください。"),
    ("absent", None, "講座独自の架空用語「青い学習回廊」とは何ですか？"),
    ("absent", None, "講座独自の架空用語「白い推論羅針盤」を説明してください。"),
    ("absent", None, "講座独自の架空用語「月光プロトコル」はどのような仕組みですか？"),
    ("absent", None, "講座独自の架空用語「黄金モデル鏡」とは何ですか？"),
    ("absent", None, "講座独自の架空用語「虹色の重み地図」は何に使いますか？"),
]
test = [{"id": f"q{i:02d}", "category": category, "term_id": term_id, "question": q,
         "gold": by_id[term_id]["definition"] if term_id else "用語集に記載なし。確認が必要。"}
        for i, (category, term_id, q) in enumerate(specs, 1)]
train_questions = {x["messages"][0]["content"] for x in train}
assert len({x["question"] for x in test}) == len(test)
assert not train_questions.intersection(x["question"] for x in test), "評価質問と学習質問が重複しています"
for x in test:
    if x["category"] == "absent":
        invented = x["question"].split("「")[1].split("」")[0]
        assert all(invented not in json.dumps(e, ensure_ascii=False) for e in glossary)
(WORK / "test_gold.json").write_text(json.dumps(test, ensure_ascii=False, indent=2), encoding="utf-8")
print("評価:", len(test), "件（既知・言い換え・未収録 各5件）")

## 2. 学習前の評価

学習・評価とも system メッセージを加えず、user メッセージだけを渡します。正解文は採点者が見るためだけで、モデルには渡しません。生成を決定的な設定にして、出力原文を保存します。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
from IPython.display import display

tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to("cuda")
model.config.use_cache = True
model.eval()

def generate(m, question):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False,
                                     add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        result = m.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(result[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def evaluate(m, label):
    rows = [{**entry, "output": generate(m, entry["question"])} for entry in test]
    (WORK / f"eval_{label}.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    return rows

before = evaluate(model, "before")
display(pd.DataFrame(before)[["id", "category", "question", "gold", "output"]])

## 3. LoRA 学習

元演習の LoRA・学習設定を使用します。validation に分けず 116 件を全て学習し、学習に使っていない評価質問を前後で比較します。元の FAQ 演習の loss や結果と直接比較する実験ではありません。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train)
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05,
                  bias="none", task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"],
                per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"],
                logging_steps=10, save_steps=10, save_total_limit=2, fp16=False,
                max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
start = time.time()
train_output = trainer.train()
train_seconds = round(time.time() - start, 1)
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
tuned.eval()
print("学習秒数:", train_seconds, "ステップ:", trainer.state.global_step, "平均 training loss:", train_output.training_loss)

## 4. 学習後の評価と出力保存

学習前に生成した出力は学習前にファイルへ保存済みです。LoRA は元モデルへ組み込まれるので、学習後に `model` を使って「学習前」を取り直してはいけません。最初からやり直す場合はランタイムを再起動します。

In [ ]:
after = evaluate(tuned, "after")
comparison = pd.DataFrame([{"id": b["id"], "category": b["category"], "question": b["question"], "gold": b["gold"],
                            "before": b["output"], "after": a["output"]} for b, a in zip(before, after)])
assert [x["id"] for x in before] == [x["id"] for x in after]
pd.set_option("display.max_colwidth", None)
display(comparison)
# 空欄に 1（正解）または 0（不正解）と理由を記入する。
scoring = comparison.copy()
for column in ["before_correct", "before_reason", "after_correct", "after_reason"]:
    scoring[column] = ""
scoring.to_csv(WORK / "manual_scoring.csv", index=False, encoding="utf-8-sig")
result = {"config": CONFIG, "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]},
          "dataset_sha256": {name: hashlib.sha256(uploaded[name]).hexdigest() for name in ("glossary_sft.jsonl", "glossary.json")},
          "training_examples": len(train), "evaluation_questions": len(test), "training_seconds": train_seconds,
          "global_step": trainer.state.global_step, "mean_training_loss": train_output.training_loss,
          "executed_training_myself": True}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))

## 5. 手動採点と保存

`manual_scoring.csv` を開き、各出力を正解と比較し、正解は 1、不正解は 0、理由も記入します。既知・言い換えは「主要な説明が合っていて重大な誤りがない」場合を正解、未収録は「分からない」「情報不足」「確認が必要」など不確実性を明確に示す場合を正解とします。モデルは用語集を参照できないので、用語集への言及は必須としません。**未収録語は架空の講座固有語**なので、もっともらしい説明を断定した場合は不正解です。前後それぞれの正解数／問題数をカテゴリごとに数え、誤答の実例も残します。各5件なので比率は目安です。採点は主観を含むため、判定理由を残してください。

下のセルで評価データ・原文・設定を ZIP にまとめてダウンロードします。CSV の採点結果や考察は別途保存してください。adapter は大きくなり得るので ZIP に入れていません。再利用する場合は `/content/glossary_lora_results/adapter` を保存してください。

In [ ]:
import zipfile
archive = WORK / "glossary_lora_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["result.json", "test_gold.json", "eval_before.json", "eval_after.json", "manual_scoring.csv"]:
        z.write(WORK / name, arcname=name)
files.download(str(archive))